# Robodog Natural-Language LLM Control

Welcome! In this notebook you will control the robot dog by typing everyday **English** (or Indonesian) commands such as `"Move forward for 5 seconds"`, `"Turn left at 20 Hz"`, or `"Sit down"` right inside Jupyter. An AI **Large Language Model (LLM)** translates your sentence into a friendly reply plus a structured robot command (`payload`, `frequency`, and `duration`), prints the exact JSON request body in the console for transparency, and politely refuses impossible or unsafe requests (like `"Fly to the ceiling"`) without moving the robot.

### Key Concepts

- **What Is an LLM, System Prompt, Few-Shot Examples, and Temperature?**
  - **LLM (`qwen3.5:9b`)**: A text-understanding AI model that converts plain sentences into structured robot numbers.
  - **System Prompt (`SYSTEM_PROMPT`)**: The AI's instruction manual telling it what the robot can do, which speed, `frequency`, and `duration` values to output, and how to refuse invalid requests.
  - **Few-Shot Examples (`FEW_SHOT_EXAMPLES`)**: Sample user-and-assistant conversations shown before your command so the AI follows the exact output format reliably.
  - **Temperature (`0.0`)**: A "creativity" dial from `0.0` to `1.0+`. Setting `TEMPERATURE = 0.0` makes responses consistent and repeatable—essential when driving a physical robot!
- **What Are an API, API Key (`Bearer`), HTTP `POST`, and JSON Payload?**
  - **API** *(Application Programming Interface)*: Like a receptionist waiting for standardized order slips. This notebook calls **two** APIs:
    1. **Cloud LLM API (`LLM_URL`)**: Public AI endpoint (`https://llm.fawzqi.my.id/v1/chat/completions`) accessed with an **API Key** (`Authorization: Bearer b4workshop`, like a workshop passkey).
    2. **Local Robot API (`API_URL`)**: The server on the robot bridge (`/api/topics/publish`) that passes commands to the robot's motors.
  - **HTTP `POST` Request & JSON Request Body**: Sending a robot command uses an **HTTP `POST`** request carrying a **JSON** order slip (curly braces `{...}` of key-value pairs):
    ```json
    {
      "topic": "/cmd_vel",
      "type": "geometry_msgs/Twist",
      "payload": {"linear": {"x": 1.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}},
      "frequency": 10,
      "duration": 5
    }
    ```
    - **`frequency`** *(in Hz)*: How many times per second the bridge should publish the command (e.g. `10` = 10 times per second).
    - **`duration`** *(in seconds)*: How long the bridge should keep publishing that command (e.g. `5` = 5 seconds).
- **What Are an IP Address, a Port, and `localhost`?**
  - **IP Address** (e.g., `192.168.1.100`): The robot's "street address" on your Wi-Fi network.
  - **Port** (e.g., `:3000`): The "room/door number" on that device where the robot API listens.
  - **`localhost`**: A special nickname meaning **"this exact computer I am typing on right now"** (used when testing against a bridge/simulator on your laptop).
- **What Is CORS and Why Don't We Need Tricks Here?**
  - **CORS** *(Cross-Origin Resource Sharing)* is a **web browser security rule** that restricts web pages from calling local servers. Because this local Jupyter notebook runs Python directly on your laptop and sends standard `Content-Type: application/json` + `Accept: application/json` requests straight from Python (`urllib.request`), CORS restrictions do not apply and no browser tricks are needed.
- **Direct Local Network Access**: Your laptop connects straight over your workshop Wi-Fi to `http://localhost:3000/...` or `http://192.168.x.x:3000/...`!


## Block 1: Setup & Environment Check

### Python 3.11.9 & `.venv` Setup
If you haven't set up Python **3.11.9** and `.venv` yet, follow **Part 0 of `hand_pose_local.ipynb`** first (Windows installer from python.org → `py -3.11 -m venv .venv` → activate → `pip install jupyter` → `jupyter notebook`).
- **Good news:** `pip install jupyter` already includes `ipywidgets` for the interactive chat block in Block 2, and the rest of this notebook uses Python's **built-in standard library** (`json`, `re`, `urllib`), so no extra `pip install` packages are needed!

### Robot Wi-Fi & `API_URL` Checklist
1. Make sure your laptop has internet access (for `LLM_URL`) **or** access to both the LLM endpoint and the robot's Wi-Fi network.
2. Ask your instructor for the robot's **IP address** (or use `localhost` for a local bridge/simulator).
3. In **Step 1.1** below, replace only the host in `API_URL` (`localhost` → e.g. `192.168.1.100`). Keep `http://`, `:3000`, and `/api/topics/publish` exact.

Run each code box with **`Shift + Enter`** (or click **▶ Run**) and wait until `[*]:` turns into a number before continuing.


In [9]:
import sys

print("Python version   :", sys.version.split()[0])
print("Python executable:", sys.executable)

if sys.version_info[:2] != (3, 11):
    print("WARNING: Expected Python 3.11.x (recommended 3.11.9).")

if sys.prefix == sys.base_prefix:
    print("WARNING: Not running inside a virtual environment (.venv). Activate .venv before launching jupyter notebook.")
else:
    print("Virtual env active:", sys.prefix)

try:
    import ipywidgets
    print(f"  [OK] ipywidgets ({ipywidgets.__version__})")
except Exception as exc:
    print(f"  [MISSING] ipywidgets: {exc}")
    print("  Install it inside .venv with: pip install ipywidgets")

print("Standard library modules (json, re, urllib) ready!")

Python version   : 3.11.9
Python executable: /mnt/volume-ssd-ubuntu-2/projects/cenim/robodog-pose-gesture/notebooks/local/.venv/bin/python
Virtual env active: /mnt/volume-ssd-ubuntu-2/projects/cenim/robodog-pose-gesture/notebooks/local/.venv
  [OK] ipywidgets (8.1.9)
Standard library modules (json, re, urllib) ready!


### Step 1.1: Configure LLM Endpoint & Robot API Settings

Edit **`API_URL`** below if your robot is at a Wi-Fi IP address (for example `API_URL = "http://192.168.1.100:3000/api/topics/publish"`), then run this cell. `DEFAULT_FREQUENCY` (`10` Hz) and `DEFAULT_DURATION` (`2` s) act as safe fallbacks if a numeric value is missing.


In [10]:
# =============================================================================
# 1. CONFIGURATION (Edit these values)
# =============================================================================

LLM_URL = "https://llm.fawzqi.my.id/v1/chat/completions"
LLM_API_KEY = "b4workshop"
MODEL_ID = "qwen3.5:9b"
REASONING_EFFORT = "none"
TEMPERATURE = 0.0

API_URL = "http://localhost:3000/api/topics/publish"  # <-- Replace 'localhost' with robot's Wi-Fi IP if needed (e.g. 192.168.1.100)
CMD_VEL_TOPIC = "/cmd_vel"
CMD_VEL_TYPE = "geometry_msgs/Twist"
DEFAULT_FREQUENCY = 10  # Default publish rate in Hz when not specified
DEFAULT_DURATION = 2    # Default duration in seconds when not specified

print("Configuration ready. Robot API_URL:", API_URL)

Configuration ready. Robot API_URL: http://localhost:3000/api/topics/publish


### Step 1.2: Engineer Your System Prompt & Few-Shot Examples

Customize how the AI translates commands into robot speeds, **`frequency`** (Hz), and **`duration`** (seconds):
- **`SYSTEM_PROMPT`**: Defines the valid actions, coordinate speeds (`linear.x` forward/back, `linear.y` left/right, `angular.z` turn, `linear.z` stand/sit, `0.0` stop), `frequency` and `duration` rules, and how to refuse impossible commands without outputting JSON.
- **`FEW_SHOT_EXAMPLES`**: Gives the AI sample commands (including explicit durations like `"for 5 seconds"` and frequencies like `"at 20 Hz"`) so it outputs `{"payload": ..., "frequency": ..., "duration": ...}` reliably.

Run this cell (and re-run it anytime you edit the prompt or examples).


In [ ]:
# =============================================================================
# 2. SYSTEM PROMPT ENGINEERING & FEW-SHOT EXAMPLES
# =============================================================================

# TODO: Engineer your system prompt here.
# Customize movement speeds, frequency/duration defaults, and response style.
SYSTEM_PROMPT = """
You are a friendly assistant that controls a robot dog (robodog).
Translate the user's command into a JSON command containing a ROS geometry_msgs/Twist payload, frequency (in Hz), and duration (in seconds).

Response rules:
1. If the command contains a VALID action (forward, backward, move/strafe left, move/strafe right, turn/rotate left, turn/rotate right, stand/stand up, sit/sit down, stop/halt/freeze):
   - On the first line, reply with a short, friendly confirmation in English (for example: "Understood, moving forward."). If the user writes in Indonesian, you may reply in Indonesian.
   - On the next line, output EXACTLY ONE valid JSON object with NO markdown code fences and NO extra text after the JSON:
     {"payload": {"linear": {"x": 0.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}, "frequency": 10, "duration": 2}
   - Velocity standard conventions inside "payload":
     * forward / move forward / walk forward: linear.x = 1.0
     * backward / move backward / walk back: linear.x = -1.0
     * move left / strafe left / step left: linear.y = 1.0
     * move right / strafe right / step right: linear.y = -1.0
     * turn left / rotate left / spin left: angular.z = 0.5
     * turn right / rotate right / spin right: angular.z = -0.5
     * stop / halt / freeze / stay still: all values 0.0
   - Velocity values are in meters per second (m/s) for linear and radians per second (rad/s) for angular. The acceptable range is between -1.0 and 1.0 for linear, and between -1.0 and 1.0 for angular.
   - Distance calculation: if the user specifies a distance (e.g. "move forward 2 meters"), calculate the duration as distance / speed (e.g. 2 meters / 1 m/s = 2 seconds). If no distance is specified, use the default duration.
   - Timing conventions:
     * "frequency": publish rate in Hz as a positive number (default 10; use the user's requested Hz if specified, e.g. "at 20 Hz" -> 20).
     * "duration": publish duration in seconds as a positive number (default 2 for walking/turning actions, 1 for stop/sit/stand; use the user's requested duration if specified, e.g. "for 5 seconds" -> 5).
   - 
2. If the command is INVALID, unsafe, impossible for a robot dog, or unclear:
   - Do NOT output any JSON at all.
   - Reply politely with: "I cannot do that." (or in the user's language).
""".strip()

# TODO: Add your few-shot examples here.
# Each entry pairs a sample "user" message with the expected "model" reply.
FEW_SHOT_EXAMPLES: list[dict[str, str]] = [
    {
        "user": "Move forward",
        "model": 'Understood, moving forward.\n{"payload": {"linear": {"x": 1.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}, "frequency": 10, "duration": 2}',
    },
    {
        "user": "Move forward for 5 seconds",
        "model": 'Understood, moving forward for 5 seconds.\n{"payload": {"linear": {"x": 1.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}, "frequency": 10, "duration": 5}',
    },
    {
        "user": "Move forward 3 meters",
        "model": 'Understood, moving forward 3 meters.\n{"payload": {"linear": {"x": 1.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}, "frequency": 10, "duration": 3}',
    },
    {
        "user": "Walk backward",
        "model": 'Understood, walking backward.\n{"payload": {"linear": {"x": -1.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}, "frequency": 10, "duration": 2}',
    },
    {
        "user": "Strafe left",
        "model": 'Understood, moving to the left.\n{"payload": {"linear": {"x": 0.0, "y": 1.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}, "frequency": 10, "duration": 2}',
    },
    {
        "user": "Move to the right",
        "model": 'Understood, moving to the right.\n{"payload": {"linear": {"x": 0.0, "y": -1.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}, "frequency": 10, "duration": 2}',
    },
    {
        "user": "Turn left",
        "model": 'Understood, turning left.\n{"payload": {"linear": {"x": 0.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.5}}, "frequency": 10, "duration": 2}',
    },
    {
        "user": "Turn left for 3 seconds at 20 Hz",
        "model": 'Understood, turning left for 3 seconds at 20 Hz.\n{"payload": {"linear": {"x": 0.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.5}}, "frequency": 20, "duration": 3}',
    },
    {
        "user": "Turn right",
        "model": 'Understood, turning right.\n{"payload": {"linear": {"x": 0.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": -0.5}}, "frequency": 10, "duration": 2}',
    },
    {
        "user": "Stop",
        "model": 'Understood, stopping now.\n{"payload": {"linear": {"x": 0.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}, "frequency": 10, "duration": 1}',
    },
    {
        "user": "Fly up into the sky",
        "model": "I cannot do that.",
    },
    {
        "user": "Jump over the table",
        "model": "I cannot do that.",
    },
    {
        "user": "Chase that person",
        "model": "I cannot do that.",
    },
]

print(f"Prompt ready with {len(FEW_SHOT_EXAMPLES)} few-shot examples.")

Prompt ready with 13 few-shot examples.


### Step 1.3: Load LLM Client, JSON Extractor & Robot Publisher Helpers

Run this cell to load the helper functions:
- **`build_messages()`**: Packs `SYSTEM_PROMPT`, `FEW_SHOT_EXAMPLES`, and your typed command into an ordered chat list.
- **`ask_llm()`**: Calls `LLM_URL` via HTTP `POST` (`Authorization: Bearer ...`) and returns the AI's raw reply.
- **`split_reply_and_twist()`**: Extracts the friendly reply along with `payload` (Twist speeds), `frequency` (Hz), and `duration` (seconds)—or returns `None` when the command was refused.
- **`build_robot_body()` / `publish_twist_payload()`**: Assembles `{"topic": ..., "type": ..., "payload": ..., "frequency": ..., "duration": ...}` and sends it directly from Python to `API_URL` with clean `Content-Type: application/json` and `Accept: application/json` headers.


In [12]:
import json
import re
import urllib.error
import urllib.request


def build_messages(
    user_text: str,
    system_prompt: str = SYSTEM_PROMPT,
    examples: list[dict[str, str]] = FEW_SHOT_EXAMPLES,
) -> list[dict[str, str]]:
    """Build chat messages with system prompt, few-shot examples, and user input."""
    messages = [{"role": "system", "content": system_prompt}]
    for example in examples:
        messages.append({"role": "user", "content": example["user"]})
        messages.append({"role": "assistant", "content": example["model"]})
    messages.append({"role": "user", "content": user_text})
    return messages


def ask_llm(
    user_text: str,
    model_id: str = MODEL_ID,
    llm_url: str = LLM_URL,
    api_key: str = LLM_API_KEY,
    reasoning_effort: str = REASONING_EFFORT,
    temperature: float = TEMPERATURE,
    timeout: float = 30.0,
) -> str:
    """Send a chat completion request to the LLM endpoint and return assistant text."""
    body = json.dumps({
        "model": model_id,
        "reasoning_effort": reasoning_effort,
        "temperature": temperature,
        "messages": build_messages(user_text),
    }).encode("utf-8")
    req = urllib.request.Request(
        llm_url,
        data=body,
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
            "User-Agent": "robodog-workshop/1.0",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    return str(data["choices"][0]["message"]["content"]).strip()


def _parse_positive_number(value, default: int | float) -> int | float:
    """Parse value as positive int (when whole) or float, else fall back to default."""
    if isinstance(value, bool) or value is None:
        return default
    try:
        num = float(value)
    except (TypeError, ValueError):
        return default
    if num <= 0:
        return default
    return int(num) if num.is_integer() else num


def split_reply_and_twist(text: str) -> tuple[str, dict | None]:
    """Separate user-facing reply text from an embedded Twist/timing JSON command."""
    decoder = json.JSONDecoder()
    idx = 0
    while (start := text.find("{", idx)) != -1:
        try:
            data, end_offset = decoder.raw_decode(text[start:])
        except json.JSONDecodeError:
            idx = start + 1
            continue

        if isinstance(data, dict):
            raw_twist = data.get("payload") if isinstance(data.get("payload"), dict) else data
            if isinstance(raw_twist.get("linear"), dict) and isinstance(raw_twist.get("angular"), dict):
                twist = {
                    section: {axis: float(raw_twist[section].get(axis, 0.0)) for axis in ("x", "y", "z")}
                    for section in ("linear", "angular")
                }
                frequency = _parse_positive_number(data.get("frequency"), DEFAULT_FREQUENCY)
                duration = _parse_positive_number(data.get("duration"), DEFAULT_DURATION)
                end = start + end_offset
                without_json = (text[:start] + text[end:]).strip()
                cleaned = re.sub(r"```(?:json)?\s*```", "", without_json).strip()
                return (cleaned or "Understood."), {
                    "payload": twist,
                    "frequency": frequency,
                    "duration": duration,
                }

        idx = start + 1

    return (text.strip() or "I cannot do that."), None


def build_robot_body(
    payload: dict[str, dict[str, float]],
    frequency: int | float = DEFAULT_FREQUENCY,
    duration: int | float = DEFAULT_DURATION,
) -> dict:
    """Wrap a Twist payload, frequency (Hz), and duration (s) into the REST publish body."""
    return {
        "topic": CMD_VEL_TOPIC,
        "type": CMD_VEL_TYPE,
        "payload": payload,
        "frequency": frequency,
        "duration": duration,
    }


def publish_twist_payload(
    payload: dict[str, dict[str, float]],
    frequency: int | float = DEFAULT_FREQUENCY,
    duration: int | float = DEFAULT_DURATION,
    api_url: str = API_URL,
    timeout: float = 1.0,
) -> tuple[bool, str]:
    """POST a Twist JSON request body with frequency and duration to the robot API."""
    body = json.dumps(build_robot_body(payload, frequency=frequency, duration=duration)).encode("utf-8")
    req = urllib.request.Request(
        api_url,
        data=body,
        headers={
            "Content-Type": "application/json",
            "Accept": "application/json",
        },
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return True, f"HTTP {resp.status}"
    except (urllib.error.URLError, TimeoutError, OSError) as exc:
        return False, f"HTTP error: {exc}"


print("Helpers loaded and ready.")

Helpers loaded and ready.


## Block 2: Running (Interactive Chat & Live Robot Control)

Run the cell below (`Shift + Enter`) to open the **interactive chat column** right inside this notebook block!

### How to Use the Chat Block
1. An interactive chat log, text input box, **Send** button, and **Clear** button appear right below the cell.
2. Type a command into the chat box and press **Enter** (or click **Send**)—for example:
   - **Valid commands:** `Move forward for 5 seconds`, `Turn left for 3 seconds at 20 Hz`, `Sit down`, `Stop`
   - **Invalid test:** `Fly to the ceiling` *(should refuse cleanly and send nothing to the robot)*
3. Look at the chat log lines after each command:
   - **`You: ...`**: Your command.
   - **`Robot: ...`**: Friendly confirmation (with JSON hidden from the text reply).
   - **`Payload: {"topic": "/cmd_vel", ..., "frequency": ..., "duration": ...}`**: Exact JSON request body sent to the robot (only printed for valid commands).
   - **`Published to <API_URL> -> HTTP 200`**: Confirmation that Python delivered the HTTP `POST` request to the robot.
4. Click **Clear** anytime to reset the chat history.

### Beginner Troubleshooting Guide
- **`LLM error: ...`**: Check your internet connection and verify `LLM_URL` and `LLM_API_KEY` in Step 1.1, then re-run Step 1.1–1.3.
- **`Published to ... -> HTTP error: ...`**:
  1. Make sure the robot dog (and its REST API bridge on port `3000`) is powered on.
  2. Confirm your laptop is on the **same Wi-Fi** as the robot.
  3. Check the IP/port in `API_URL` (Step 1.1), and remember to **re-run Step 1.1** and Block 2 after changing it!
- **AI misunderstands a command or picks the wrong speed/timing**: Adjust `SYSTEM_PROMPT` or add an example in `FEW_SHOT_EXAMPLES` (Step 1.2), then re-run Step 1.2, Step 1.3, and Block 2.


In [14]:
import html
import warnings
import ipywidgets as widgets
from IPython.display import display


def build_chat_ui() -> widgets.VBox:
    """Create an interactive Jupyter chat column with scrollable log and send/clear controls."""
    status_bar = widgets.HTML("<b>Status:</b> Ready! Type a robot command below.")
    chat_log = widgets.HTML()
    chat_input = widgets.Text(
        placeholder="Type a robot command (e.g. Move forward for 5 seconds, Turn left at 20 Hz)...",
        layout=widgets.Layout(flex="1 1 auto", width="auto"),
    )
    send_btn = widgets.Button(description="Send", button_style="primary", layout=widgets.Layout(width="90px"))
    clear_btn = widgets.Button(description="Clear", layout=widgets.Layout(width="80px"))

    messages_html: list[str] = []

    def render_log() -> None:
        chat_log.value = (
            '<div style="background:#181818; color:#eee; font-family:monospace; font-size:13px; '
            'padding:12px; height:300px; overflow-y:auto; border:1px solid #444; border-radius:4px; '
            'display:flex; flex-direction:column-reverse;"><div>'
            + "".join(messages_html)
            + "</div></div>"
        )

    def append_log(text: str, color: str = "#eee") -> None:
        safe = html.escape(text)
        messages_html.append(
            f'<div style="color:{color}; margin:3px 0; white-space:pre-wrap; word-break:break-word;">{safe}</div>'
        )
        render_log()

    def handle_send(_=None) -> None:
        user_text = chat_input.value.strip()
        if not user_text:
            return
        chat_input.value = ""
        send_btn.disabled = True
        status_bar.value = "<b>Status:</b> Thinking..."
        append_log(f"You: {user_text}", "#90caf9")

        try:
            reply = ask_llm(user_text)
            user_message, command = split_reply_and_twist(reply)
            append_log(f"Robot: {user_message}", "#a5d6a7")

            if command is not None:
                request_body = build_robot_body(
                    command["payload"],
                    frequency=command["frequency"],
                    duration=command["duration"],
                )
                _, api_status = publish_twist_payload(
                    command["payload"],
                    frequency=command["frequency"],
                    duration=command["duration"],
                    api_url=API_URL,
                )
                append_log(f"Payload: {json.dumps(request_body)}", "#ffe082")
        except (urllib.error.URLError, TimeoutError, OSError, KeyError, IndexError, json.JSONDecodeError) as exc:
            append_log(f"LLM error: {exc}", "#ef9a9a")
        finally:
            send_btn.disabled = False
            status_bar.value = "<b>Status:</b> Ready! Type a robot command below."

    def handle_clear(_=None) -> None:
        messages_html.clear()
        render_log()

    send_btn.on_click(handle_send)
    clear_btn.on_click(handle_clear)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        chat_input.on_submit(handle_send)

    render_log()
    controls = widgets.HBox([chat_input, send_btn, clear_btn], layout=widgets.Layout(gap="8px", margin="8px 0 0 0"))
    return widgets.VBox([status_bar, chat_log, controls], layout=widgets.Layout(max_width="780px"))


display(build_chat_ui())